# Selectividad Exams Parser

This script reads the content of the PDF files, to extract the information from files

## Libraries import

### Required packages for OCR processing:
**Ubuntu based:**
- tesseract-ocr
- tesseract-ocr-spa
- poppler-utils

**Archlinux based:**
- tesseract
- tesseract-data-spa

In [ ]:
# !pip install pdf2image
# !pip install pytesseract

In [ ]:
from datetime import datetime
from multiprocessing import Pool
from pathlib import Path
import os
import shutil

import pandas as pd
from pdf2image import convert_from_path
import pytesseract

In [ ]:
# set root path
root = "/home/daniel/code/emestrada_parser/"
os.chdir(root)

## Functions

### *pdf_to_text*

Extract the text from pdf files, page by page

In [ ]:
def pdf_to_text(file: Path) -> list:
    """
    Processes a single PDF file to extract its content as text.
    """
    images = convert_from_path(file)[1:]
    # parse each page
    pages = []
    for i, page in enumerate(images):
        pages.append(pytesseract.image_to_string(page, lang='spa'))
    return pages

### *extract_statement*
Extract the statement from a page in text format

In [ ]:
def extract_statement(page: str) -> str:
    # split the content into lines
    text = page.split('\n')

    # list to store the processed lines
    extracted_text = []
    # start from the 4th line
    for line in text[4:]:

        # check the first word of the line
        first_word = line.replace('.', '').split(' ')[0].strip().upper()

        # stop at examen details line
        if first_word in ['SOCIALES', 'FISICA', 'MATES',
                          'QUÍMICA', 'QUIMICA', 'QUIÍMICA']:
            extracted_text.append(line.strip())
            break
        
        # add the line to final_text
        else:
            if not line == '':
                extracted_text.append(line.strip())

    # join the list into a single string
    extracted_text = '\n'.join(extracted_text)

    return extracted_text

### *extract_exam_details*
Extract the exam details, like subject, year, exam and exercise number.

In [ ]:
def extract_exam_details(statement : str) -> dict :
    # get last line with exam details
    lines = [i for i in statement.split('\n') if i != '']
    exam_details = lines[-1].lower()

    # strip whitespaces
    exam_details = (exam_details.replace('.', '')            
                    .replace(',', '')
                    .replace('(', '')
                    .replace(')', '').strip()
    )
    
    # split the words by ' '
    exam_details = exam_details.split(' ')

    # MATES CCSS
    # SOCIALES II. 2017 JUNIO. EJERCICIO 2. OPCIÓN A
    # 
    if exam_details[0].startswith('sociales'):
        exam_details[0] = 'MATES_CCSS'
        # remove the second element in the list
        del exam_details[1]
    
       
    # exam dictionary
    details = ['subject', 'year', 'exam', 'exercise']
    exam_dict: dict[str, str | int | None] = dict.fromkeys(details)
    exam_dict = {k:None for k in details}

    ## fill dictionary with values
    
    # subject
    exam_dict['subject'] = exam_details[0]
    
    # year
    # SOCIALES II. PONENCIA 2009. EJERCICIO 2
    if exam_details[1] == 'ponencia':
        # switch ponencia and year
        exam_details[1], exam_details[2] = exam_details[2], exam_details[1]

    exam_dict['year'] = int(exam_details[1])

    # parse the exam string
    if exam_details[2].startswith('reserva'):
        exam_dict['exam'] = ' '.join(exam_details[2:4]).title()
        exam_index_start = 4
    else:
        exam_dict['exam'] = exam_details[2].title()
        exam_index_start = 3
    
    exam_dict['exercise'] = ' '.join(exam_details[exam_index_start:]).title()

    
    return exam_dict

### *generate_error_log*
generate a error log and add it to errors_timestamp.log file

In [ ]:
def generate_error_log(file, page, error_type, statement = None, details = None):
    log = f'Error processing {error_type}, in {file.name} at page {page+2}\n'
    
    # add timestamp to log file
    current_datetime = datetime.now()
    formatted_dt = current_datetime.strftime("%d-%m-%Y")
    with open(f'./error_logs/errors_{formatted_dt}.log', 'a') as f:
        f.write(log)
        f.write('*' * 10 + '\n')
        if statement:
            f.write(f'{statement}\n')
        elif details:
            f.write(f'{details}\n')
        f.write('*' * 10 + '\n')
    
    # print log to console without the new line
    print(log[:-2])

### *process_file*
Process a pdf file, extracting its information: statement and exam information 

In [ ]:
def process_file(file: Path) -> pd.DataFrame:
    
    topic = file.stem.split(' - ')[-1]   
    
    # create a dict to store parsed values
    exercises_dict = dict.fromkeys(['subject', 'year', 'exam', 'exercise', 'statement'])
    # initialize the dict with empty lists
    exercises_dict = {k:[] for k in exercises_dict.keys()}

    pages = pdf_to_text(file)

    # parse each page in the pdf file

    print(f'Success parsing file: {file.stem}')
    
    for index, page in enumerate(pages):

        # extract statement from page
        try:
            statement = extract_statement(page)
        # error with extracting statement
        except:
            for line in statement.split('\n'):
                if line.startswith('www.emestrada.org'):
                    continue
            generate_error_log(file = file, page = index, error_type= 'statement', statement=page)
            continue

        # extract exam details
        try:
            details = extract_exam_details(statement)

            # store the details in the exercises dictionary
            for k,v in details.items():
                exercises_dict[k].append(v)
            
            # finally, add the statement text 
            exercises_dict['statement'].append(statement)
        
        # error with extracting details
        except:
            # pages ending with www.emestrada.org are pages with solution to exercise
            if not statement.endswith('www.emestrada.org'):
                generate_error_log(file = file, page = index, error_type= 'details', details = statement)
                output = f'File {file.stem} parsed but with errors'
            continue
        
    df = pd.DataFrame(exercises_dict)
    df['topic'] = topic
    df = df[["subject", "topic", "year", "exam", "exercise", "statement"]]

    return df

In [ ]:
#process_file(Path("./pdf_files/quimica/enlace_quimico/2011 - Enlace Químico.pdf"))

### *process_folder*
Processes all files within a folder

In [ ]:
def process_folder(folder_path:Path) -> pd.DataFrame:
    df = pd.DataFrame()
    
    
    # get the list of pdf files within the folder
    files = sorted( [i for i in folder_path.iterdir()
                        if i.suffix == '.pdf'] )
    
    # process each pdf file to extract its information
    for file in files:
        try:
            exercises_df = process_file(file)
            # each file contains a dataframe with columns:
            # subject, year, exam, exercise, topic
            df = pd.concat([df, exercises_df], axis = 0)

        except:
            pass
        
    # correct wrong subjects
    df["subject"] = df["subject"].apply(lambda x: 'Química'
                                   if x.endswith('mica') else x)
    
    # create csv folder to store csv files
    ## concatenate folder path provided with a subfolder called csv_files
    csv_folder = folder_path / Path('csv_files')
    if not csv_folder.exists():
        csv_folder.mkdir()
        print(f"Folder '{csv_folder}' created.")
    else:
        print(f"Folder '{csv_folder}' already exists.")
        
    # export the output of the processed pdf files to a csv file
    output_file = csv_folder / Path(f'exercises_{folder_path.stem}.csv')
    
    df.to_csv(output_file, index=False)
    
    return df

In [ ]:
process_folder(Path("pdf_files/quimica/termoquimica").resolve())

## Process pdf files

In [ ]:
# folders = sorted([i.resolve() for i in Path('./pdf_files/quimica/').iterdir() if i.is_dir()])
# folders

In [ ]:
# for f in folders:
#     process_folder(f)

## Gather the csv files and combine them

### Get the path of all of the csv files:

In [ ]:
# csv_files_paths = list(Path('pdf_files/quimica/').rglob('*.csv'))
# csv_files_paths

### Move them to a csv folder to gather them all into a unified folder.

In [ ]:
# csv_folder = root / Path("csv_statements")
# csv_folder.mkdir(exist_ok=True)

# for f in csv_files_paths:
#     shutil.move(f, csv_folder / f.name)

### Combine into one large csv file

In [ ]:
# os.chdir("./csv_statements")

In [ ]:
# csv_files = [i.name for i in Path(".").iterdir()]
# df = pd.DataFrame()

# for f in csv_files:
#     df = pd.concat([df, pd.read_csv(f)], axis=0)

### Review the dataframe with the combined csv files

In [ ]:
# df.head()

In [ ]:
# df.subject = "química"

In [ ]:
# df.to_csv("quimica_statements.csv", index=False)